# Run the scripts within VIB_10xmultiome_2_WS3000F folder  first 
Particularly, run VIB_10xmultiome_2_WS3000F/_ArchR_TSS/prepare_frag_for_archr.sh

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D 
from matplotlib import gridspec
import matplotlib.colors as colors


import os
import seaborn as sns


## Load RNA cell Barcodes from RNA-only modality CR analysis

In [3]:
data_set_name = 'human_brain_3k'

entropy_dir ='/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F/'

entropy_all_file = os.path.join(entropy_dir, 'calculated_barcode_entropy_wDebrisflag_df.tsv')
entropy_df = pd.read_csv(entropy_all_file, sep=',', index_col=0)
entropy_df['atac_bc'] = entropy_df.index.str.split('-').str[0]
print(entropy_df.shape) 
entropy_df.head(2)

(33287, 8)


,Entropy,Entropy_open_region,Mp,mle_lambda,P0_open_region,P_closed_state,DNA_debris,atac_bc
AAGCCATAGCTAATAG-1,0.005725,1.069145,11,0.343957,0.708959,0.999545,NO,AAGCCATAGCTAATAG
CACGGTAAGAATCACA-1,0.004039,1.910865,19,1.377521,0.252203,0.999694,NO,CACGGTAAGAATCACA


In [4]:
# Load filtered barcodes from each filtering method
sade_bcs= os.path.join(entropy_dir, 'entropy_filtered_bc_df.tsv')
cr_bcs = os.path.join(entropy_dir, '_CR_FRIP', 'barcodes.tsv')
tss_bcs = os.path.join(entropy_dir, '_ArchR_TSS', 'QualityControl/VIB_10xmultiome_2_Metadata.tsv')

sade_bc_df = pd.read_csv(sade_bcs, sep=',', index_col=0)
cr_bcs_df = pd.read_csv(cr_bcs, sep='\t', index_col=0, header=None)
cr_bcs_df.index.name = 'atac_barcodes'
tss_bcs_df = pd.read_csv(tss_bcs, sep='\t', index_col=0)
tss_bcs_df['atac_barcodes'] = tss_bcs_df.index.map(lambda x: x.split('#')[1])


In [5]:
tss_bcs_df.head(2)

,nFrags,nMonoFrags,nDiFrags,nMultiFrags,TSSEnrichment,ReadsInTSS,Keep,ReadsInPromoter,PromoterRatio,ReadsInBlacklist,BlacklistRatio,atac_barcodes
cellNames,,,,,,,,,,,,
VIB_10xmultiome_2#GGGTTGCAGCTCAAGG-1,93652,47018,37344,9290,20.827,61970,1,55796,0.297890,1417,0.007565,GGGTTGCAGCTCAAGG-1
VIB_10xmultiome_2#GTCATTTAGAGAGCAA-1,92024,24025,55290,12709,12.683,17927,1,17285,0.093916,839,0.004559,GTCATTTAGAGAGCAA-1


In [6]:
# create union cell set by combining three filtering sets
Union_cell_subdir = os.path.join(entropy_dir, '_cell_calling_comparison')
os.mkdir(Union_cell_subdir) if not os.path.exists(Union_cell_subdir) else None
tosave_union_cell_set = os.path.join(Union_cell_subdir, 'Union_cell_3set_all_info.tsv')


# Load fragments file
frag_file = os.path.join(entropy_dir, 'fragments.tsv')
frag = pd.read_csv(frag_file, sep='\t', index_col=3, header=None)
frag.index.name ='atac_bc'
frag.columns = ['chrom', 'start', 'end', 'support']
frag['atac_barcodes'] = frag.index
      

total_frag = frag.groupby('atac_barcodes').size().rename('total_fragments')
total_frag_df = total_frag.to_frame()


total_frag_df['atac_pass_CR'] = total_frag_df.index.isin(cr_bcs_df.index)
total_frag_df['atac_pass_entropy'] = total_frag_df.index.isin(sade_bc_df.index)
total_frag_df['atac_pass_archr_TSS'] = total_frag_df.index.isin(tss_bcs_df['atac_barcodes'])


total_frag_df['_2set_identified_by_atac_SC'] = total_frag_df.apply(lambda row: 'both' if row['atac_pass_CR'] and row['atac_pass_entropy'] else ('entropy_only' if row['atac_pass_entropy'] else ('cr_only' if row['atac_pass_CR'] else ('entropy_only' if row['atac_pass_entropy'] else 'none'))), axis=1)
total_frag_df['_2set_identified_by_atac_ST'] = total_frag_df.apply(lambda row: 'both' if row['atac_pass_archr_TSS'] and row['atac_pass_entropy'] else ('tss_only' if row['atac_pass_archr_TSS'] else ('entropy_only' if row['atac_pass_entropy']  else 'none')), axis=1)


total_frag_df.loc[total_frag_df['atac_pass_CR'] & total_frag_df['atac_pass_entropy'] & total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'common'
total_frag_df.loc[total_frag_df['atac_pass_CR'] & total_frag_df['atac_pass_entropy'] & ~total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'SADE+CR, not TSS'
total_frag_df.loc[~total_frag_df['atac_pass_CR'] & total_frag_df['atac_pass_entropy'] & total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'SADE+TSS, not CR'
total_frag_df.loc[total_frag_df['atac_pass_CR'] & ~total_frag_df['atac_pass_entropy'] & total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'CR+TSS, not SADE'  
total_frag_df.loc[total_frag_df['atac_pass_entropy'] & ~total_frag_df['atac_pass_CR'] & ~total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'SADE unique'
total_frag_df.loc[total_frag_df['atac_pass_CR'] & ~total_frag_df['atac_pass_entropy'] & ~total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'CR unique'
total_frag_df.loc[~total_frag_df['atac_pass_CR'] & ~total_frag_df['atac_pass_entropy'] & total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'TSS unique'
total_frag_df.loc[~total_frag_df['atac_pass_CR'] & ~total_frag_df['atac_pass_entropy'] & ~total_frag_df['atac_pass_archr_TSS'],'_3set_identified_by_atac'] = 'none'

for col in ['Entropy', 'DNA_debris', 'P_closed_state']:
    total_frag_df[col] = total_frag_df.index.map(lambda x: entropy_df.loc[x, col] if x in entropy_df.index else np.nan)

total_frag_df.to_csv(tosave_union_cell_set, sep='\t')

In [12]:
total_frag_df.head(2)

,total_fragments,atac_pass_CR,atac_pass_entropy,atac_pass_archr_TSS,_2set_identified_by_atac_SC,_2set_identified_by_atac_ST,_3set_identified_by_atac,Entropy,DNA_debris,P_closed_state
atac_barcodes,,,,,,,,,,
AAACAAGCAAACATGT-1,4,False,False,False,none,none,none,NaN,NaN,NaN
AAACAAGCAAACCAGC-1,1,False,False,False,none,none,none,NaN,NaN,NaN


## Load

In [139]:
# Load union set cells 
Union_cell_subdir = os.path.join(entropy_dir, '_cell_calling_comparison')
Union_cell_info_file = os.path.join(Union_cell_subdir, 'Union_cell_3set_all_info.tsv')
union_cell_all_info = pd.read_csv(Union_cell_info_file, sep='\t', index_col=0)


union_cell_all_info['log_entropy'] = np.log2(union_cell_all_info['Entropy'])
union_cell_all_info['log_total_fragments'] = np.log1p(union_cell_all_info['total_fragments'])


union_cell_info = union_cell_all_info.loc[union_cell_all_info['_3set_identified_by_atac'] != 'none'].copy()

union_cell_all_info.shape, union_cell_info.shape

((602721, 12), (3857, 12))

In [140]:
union_cell_info.head(2)

,total_fragments,atac_pass_CR,atac_pass_entropy,atac_pass_archr_TSS,_2set_identified_by_atac_SC,_2set_identified_by_atac_ST,_3set_identified_by_atac,Entropy,DNA_debris,P_closed_state,log_entropy,log_total_fragments
atac_barcodes,,,,,,,,,,,,
AAACAGCCATAGACTT-1,16375,True,True,True,both,both,common,0.269836,NO,0.953848,-1.889846,9.703572
AAACAGCCATTATGCG-1,1245,True,True,True,both,both,common,0.176822,NO,0.973443,-2.499632,7.127694


In [142]:
col_list = [ '_2set_identified_by_atac_SC', '_2set_identified_by_atac_ST', '_3set_identified_by_atac']
col_list += ['total_fragments']
for col in col_list:
    entropy_df[col] = entropy_df.index.map(union_cell_all_info[col]).fillna('none')

entropy_df['log_entropy'] = np.log2(entropy_df['Entropy'])
entropy_df['log_total_fragments'] = np.log1p(entropy_df['total_fragments'])
entropy_df.head(2)

,Entropy,Entropy_open_region,Mp,mle_lambda,P0_open_region,P_closed_state,DNA_debris,atac_bc,log_entropy,_2set_identified_by_atac_SC,_2set_identified_by_atac_ST,_3set_identified_by_atac,total_fragments,log_total_fragments
GTTTCAGCACCTAAGC-1,0.080550,0.287638,40,0.049590,0.951619,0.990037,NO,GTTTCAGCACCTAAGC,-3.633968,none,none,none,386,5.958425
AGCTTAATCGAGCAAA-1,0.008189,0.754153,10,0.193748,0.823866,0.999316,NO,AGCTTAATCGAGCAAA,-6.932182,none,none,none,60,4.110874


In [ ]:
frag_in_Raw_peak_file = os.path.join(Union_cell_subdir, 'Rawpeaks', 'frag_overlap/frag_in_peaks.tsv')
frag_in_raw_peaks = pd.read_csv(frag_in_Raw_peak_file, sep='\t', index_col=3, header=None)
frag_in_raw_peaks.index.name ='atac_barcodes'
frag_in_raw_peaks.columns = ['chrom', 'start', 'end', 'support']

Nfrag_in_rawpeaks = frag_in_raw_peaks.groupby('atac_barcodes').size().rename('N_frag_in_raw_peaks')

union_cell_info['Nfrag_in_rawpeaks'] = union_cell_info.index.map(Nfrag_in_rawpeaks).fillna(0)
union_cell_info['frip_rawpeaks'] = union_cell_info['Nfrag_in_rawpeaks'] / union_cell_info['total_fragments']

In [151]:

union_cell_info['Nfrag_in_rawpeaks'] = union_cell_info.index.map(Nfrag_in_rawpeaks).fillna(0)
union_cell_info['frip_rawpeaks'] = union_cell_info['Nfrag_in_rawpeaks'] / union_cell_info['total_fragments']

In [152]:
# Those in the union set but in neither of CR or SADE are uniquely identified by TSS enrichment
union_cell_info['_2set_identified_by_atac_SC_mod'] = union_cell_info['_2set_identified_by_atac_SC'].replace('none', 'tss_only')